<a href="https://colab.research.google.com/github/ChitiKatepa/FarmingDroneAI/blob/main/homemade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import time
import pandas as pd
import shutil
from pathlib import Path
import itertools
from PIL import Image
import io

import cv2
import numpy as np
import seaborn as sns
sns.set_style('darkgrid')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras import regularizers

from keras.applications import VGG16
from keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from keras.models import Model

import warnings
warnings.filterwarnings("ignore")

In [4]:
images = []
labels = []

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

data_path = Path("/content/drive/My Drive/archive/data")

for subfolder in data_path.iterdir():
    if not subfolder.is_dir():
        continue

    for image_file in subfolder.iterdir():
        if image_file.is_file():
            images.append(str(image_file))
            labels.append(subfolder.name)

data = pd.DataFrame({'image': images, 'label': labels})

Mounted at /content/drive


In [5]:
strat = data['label']
train_df, dummy_df = train_test_split(data, train_size= 0.81, shuffle= True, random_state= 123, stratify= strat)

strat = dummy_df['label']
valid_df, test_df = train_test_split(dummy_df,  train_size= 0.5, shuffle= True, random_state= 123, stratify= strat)

print("Training set shape:", train_df.shape)
print("Validation set shape:", valid_df.shape)
print("Test set shape:", test_df.shape)

#Data visualisation
batch_size = 32
img_size = (224, 224)
channels = 3
img_shape = (img_size[0], img_size[1], channels)

tr_gen = ImageDataGenerator()
ts_gen = ImageDataGenerator()

train_gen = tr_gen.flow_from_dataframe(train_df, x_col='image', y_col='label', target_size=img_size, class_mode='categorical', color_mode='rgb', shuffle=True, batch_size=batch_size)

valid_gen = ts_gen.flow_from_dataframe(valid_df, x_col='image', y_col='label', target_size=img_size, class_mode='categorical', color_mode='rgb', shuffle=True, batch_size=batch_size)

test_gen = ts_gen.flow_from_dataframe(test_df, x_col='image', y_col='label', target_size=img_size, class_mode='categorical', color_mode='rgb', shuffle=False, batch_size=batch_size)

Training set shape: (3392, 2)
Validation set shape: (398, 2)
Test set shape: (398, 2)
Found 3392 validated image filenames belonging to 4 classes.
Found 398 validated image filenames belonging to 4 classes.
Found 398 validated image filenames belonging to 4 classes.


In [6]:
#kernels of the images for training purposes
#Using a CNN
model = Sequential()

model.add(tf.keras.layers.InputLayer(input_shape=(224, 224, 3)))

model.add(Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), activation='relu'))
model.add(Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())


model.add(Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), activation='relu'))
model.add(Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())


model.add(Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), activation='relu'))
model.add(Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())
model.add(Dropout(0.25))

model.add(Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), activation='relu'))
model.add(Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())
model.add(Dropout(0.25))

model.add(Conv2D(filters=64, kernel_size=(3, 3), strides=(1, 1), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(BatchNormalization())
model.add(Dropout(0.25))

model.add(Flatten())

model.add(Dropout(0.3))

model.add(Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.0001)))


model.add(Dense(4, activation='softmax', kernel_regularizer=tf.keras.regularizers.l2(0.0001)))

#Callback and compilation of the code
class myCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if(logs.get('val_accuracy') > 0.98):
            print("\nReached 98% accuracy so cancelling training!")
            self.model.stop_training = True

callbacks = myCallback()

model.compile(optimizer=Adamax(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    train_gen,
    epochs=5,
    batch_size=32,
    verbose=1,
    validation_data=valid_gen,
    callbacks=[callbacks]
)

model.save('model2.h5')

Epoch 1/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 1828s 17s/step - accuracy: 0.7444 - loss: 0.6771 - val_accuracy: 0.5276 - val_loss: 1.1455
Epoch 2/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 1743s 16s/step - accuracy: 0.8199 - loss: 0.4765 - val_accuracy: 0.4698 - val_loss: 1.6893
Epoch 3/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 1766s 17s/step - accuracy: 0.8488 - loss: 0.3945 - val_accuracy: 0.8141 - val_loss: 0.5401
Epoch 4/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 1758s 17s/step - accuracy: 0.8703 - loss: 0.3556 - val_accuracy: 0.8090 - val_loss: 0.5440
Epoch 5/100
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 16s/step - accuracy: 0.8919 - loss: 0.3109 

KeyboardInterrupt: 